# Example 12.1 / 1: Blend Objectives with Weights

Companion notebook for **12.1 Multiple Objectives** in the AMPL lab Documentation, section "Blend Objectives with Weights". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Blend Objectives with Weights

A weighted objective adds the measures with explicit exchange rates. The weights are part of the
decision policy, not harmless numerical settings: rescaling dollars to thousands of dollars
changes their meaning unless the weight is adjusted too.

With `weight_cost` 1 and `weight_emission` 0.3, the blend favors the cheap plant. Reporting cost
in thousands of dollars (`cost_scale` 0.001) while keeping the same weights makes emissions
dominate and moves production to the clean plant. The run loop changes `cost_scale`, solves with
`ampl.solve()`, and reads the plan with `to_dict()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                           # production plants
param unit_cost {PLANTS} >= 0;        # dollars per unit
param unit_emission {PLANTS} >= 0;    # kg CO2 per unit
param capacity {PLANTS} >= 0;         # units
param demand >= 0;                    # units
param cost_scale > 0 default 1;       # 1 = dollars, 0.001 = thousands of dollars
param weight_cost >= 0;               # weight per scaled cost unit
param weight_emission >= 0;           # weight per kg CO2

var Make {p in PLANTS} >= 0, <= capacity[p];  # units produced at each plant
# cost in the units set by cost_scale
var TotalCost = cost_scale * sum {p in PLANTS} unit_cost[p] * Make[p];
var TotalEmission = sum {p in PLANTS} unit_emission[p] * Make[p];  # emissions, kg CO2

minimize Tradeoff:                    # weighted blend; same weights, different cost scales
    weight_cost * TotalCost + weight_emission * TotalEmission;

subject to MeetDemand: sum {p in PLANTS} Make[p] = demand;  # total output covers demand exactly

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["A", "B"]
ampl.param["unit_cost"] = {"A": 2, "B": 4}
ampl.param["unit_emission"] = {"A": 5, "B": 1}
ampl.param["capacity"] = {"A": 8, "B": 8}
ampl.param["demand"] = 10
ampl.param["weight_cost"] = 1
ampl.param["weight_emission"] = 0.3

In [5]:
# run.py
for scale in [1, 0.001]:
    ampl.param["cost_scale"] = scale
    ampl.solve(solver="highs", verbose=False)
    print(scale, ampl.solve_result, ampl.var["Make"].to_dict())

1 solved {'A': 8, 'B': 2}
0.001 solved {'A': 2, 'B': 8}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PLANTS := A B;
param: unit_cost unit_emission capacity :=
    A    2   5   8
    B    4   1   8 ;
param demand := 10;
param weight_cost := 1;
param weight_emission := 0.3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
for {scale in {1, 0.001}} {
    let cost_scale := scale;
    solve;
    printf "%g %s A=%g B=%g\n", scale, solve_result, Make['A'], Make['B'];
}

1 solved A=8 B=2
0.001 solved A=2 B=8
